<a href="https://colab.research.google.com/github/abdelrahmanmohamed05/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahmanmohamed05/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:

# Setup. Put HF_TOKEN in the Colab Secret/environment; never paste it into this cell.
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate) as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()
    return getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the secret out of the SQL text.
token = get_hf_token().replace("'", "''")
con.execute(f"SET VARIABLE hf_token = '{token}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
MID = MAR
DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected. Feature window: February 2026. Label window: March 2026.")


Hugging Face READ token: ··········
Connected. Feature window: February 2026. Label window: March 2026.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Features — February only (maximum five)

| Feature | Meaning | Available when? |
|---|---|---|
| `log_imp_feb` | log-transformed February GSC impressions | Knowable at the decision moment because it uses only measured February impressions. |
| `log_clk_feb` | log-transformed February GSC clicks | Knowable at the decision moment because these clicks happened before the 2026-02-28 cutoff. |
| `ctr_feb` | February clicks / impressions | Knowable at the decision moment because both numerator and denominator come only from February. |
| `pos_feb` | impression-weighted average GSC position | Knowable at the decision moment because it uses February `gsc_sum_position` and impressions only. |
| `days_with_imps_feb` | number of February days with impressions > 0 | Knowable at the decision moment because it counts only days inside the completed February window. |

### Label

`went_dark` is computed from March GSC clicks. It is an outcome, not a feature.

### Context

`client_hash_id`, `content_hash_id`, `report_date`, `month`, `gsc_data_available`, `is_published`, and `content_created_date` are used for joins, filtering, windows, or reporting.

### Deliberately excluded

- March `gsc_clicks` / `gsc_impressions`: future outcome information.
- `last_optimized_date`, `optimization_eligible_date`, and post-cutoff `content_updated_date`: not knowable at the February decision moment.
- `fact_content_query_90d`: fixed 90-day aggregates overlap the March outcome window.
- GA4/session/AI engagement fields: downstream engagement information and/or unavailable behind their own availability flags.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:

# QUERY 1 — Grain
# Claim: the raw March fact has one row per report_date × client × content.
q1 = con.sql(f"""
SELECT COUNT(*) AS duplicate_groups
FROM (
  SELECT report_date, client_hash_id, content_hash_id
  FROM {MID}
  GROUP BY 1,2,3
  HAVING COUNT(*) > 1
) t
""").df()

display(q1)
assert int(q1.loc[0, "duplicate_groups"]) == 0


,duplicate_groups
0,0


In [3]:

# QUERY 2 — March slice row count and date span
q2 = con.sql(f"""
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date,
  COUNT(DISTINCT report_date) AS distinct_dates
FROM {MID}
""").df()

display(q2)


,row_count,first_date,last_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


In [4]:

# QUERY 3 — Availability (explicit IS TRUE)
# Show how many March rows survive the GSC availability flag.
q3 = con.sql(f"""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS unavailable_rows
FROM {MID}
""").df()

display(q3)


,all_rows,available_rows,unavailable_rows
0,9841378,3611061,6230317



### Five-feature frame + March label

The feature frame below uses February only for model inputs. The March label is joined afterward as the outcome.


In [5]:

# Build the five-feature frame from February only.
feb_agg = con.sql(f"""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_impressions) AS imp_feb,
  SUM(gsc_clicks) AS clk_feb,
  SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS pos_feb,
  COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_imps_feb
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY 1,2
HAVING SUM(gsc_impressions) >= 100
   AND SUM(gsc_clicks) >= 3
""").df()

feature_meta = con.sql(f"""
SELECT client_hash_id, content_hash_id, content_type, content_created_date, is_published
FROM {DIM}
WHERE is_published IS TRUE
  AND content_created_date <= DATE '2026-02-28'
""").df()

features = feb_agg.merge(
    feature_meta,
    on=["client_hash_id","content_hash_id"],
    how="inner"
)

features["log_imp_feb"] = np.log1p(features["imp_feb"])
features["log_clk_feb"] = np.log1p(features["clk_feb"])
features["ctr_feb"] = features["clk_feb"] / features["imp_feb"]

feature_cols = [
    "log_imp_feb",
    "log_clk_feb",
    "ctr_feb",
    "pos_feb",
    "days_with_imps_feb",
]

feature_frame = features[
    ["client_hash_id","content_hash_id"] + feature_cols
].copy()

# March outcome: this is the label window, not a feature window.
march_label = con.sql(f"""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_clicks) AS clk_mar
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY 1,2
""").df()

feature_frame = feature_frame.merge(
    march_label,
    on=["client_hash_id","content_hash_id"],
    how="left"
)

feature_frame["clk_mar"] = feature_frame["clk_mar"].fillna(0)
feature_frame["went_dark"] = (feature_frame["clk_mar"] == 0).astype(int)

display(feature_frame[["client_hash_id","content_hash_id"] + feature_cols + ["went_dark"]].head(10))
print(f"feature rows: {len(feature_frame):,}")
print(f"positive rate: {feature_frame.went_dark.mean():.3f}")


,client_hash_id,content_hash_id,log_imp_feb,log_clk_feb,ctr_feb,pos_feb,days_with_imps_feb,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,6.920672,1.386294,0.002964,28.886364,28,0
1,client_e547b89c05043229,content_ccbb253f142217c3,7.377134,2.079442,0.004380,17.273467,28,0
2,client_e547b89c05043229,content_c6dbda992ad84127,7.961719,1.386294,0.001046,17.909693,28,0
3,client_e547b89c05043229,content_fe83160838cdab99,7.855932,2.302585,0.003488,17.762016,28,0
4,client_e547b89c05043229,content_c5baa3a03cfccc41,8.457868,3.091042,0.004458,10.231373,28,0
5,client_e547b89c05043229,content_516e5fb67d095bdd,7.945201,2.772589,0.005317,8.545197,28,0
6,client_e547b89c05043229,content_7c652f71478094c9,7.617268,2.397895,0.004921,6.747539,28,0
7,client_e547b89c05043229,content_d9ff089f62d1bd27,7.331715,1.945910,0.003929,21.127046,28,0
8,client_e547b89c05043229,content_8dac7504d04fdf6e,7.408531,2.197225,0.004851,16.309885,28,0
9,client_e547b89c05043229,content_b60f07578c351b40,6.823286,1.609438,0.004357,19.796296,28,0


feature rows: 29,700
positive rate: 0.051



### Deliberate leakage experiment

I will intentionally add `clk_mar`, a direct source of the March label, to the feature set. This is **not** an acceptable production feature: at 2026-02-28 the March clicks do not exist yet.

The purpose is to make the leakage failure visible: if a model can see the label source, its quick score should become unrealistically close to perfect. Then I remove the leaked column and keep the honest score.


In [6]:

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_df = feature_frame.dropna(subset=feature_cols + ["went_dark"]).copy()

X = model_df[feature_cols]
y = model_df["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=4, random_state=42)
)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

# Intentional leak: March clicks are directly derived from the outcome window.
X_leaky = model_df[feature_cols + ["clk_mar"]]
XL_train, XL_test, yL_train, yL_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=4, random_state=42)
)
leaky_model.fit(XL_train, yL_train)
leaky_score = accuracy_score(yL_test, leaky_model.predict(XL_test))

print(f"honest accuracy (5 features): {honest_score:.3f}")
print(f"leaky accuracy (+ March clk_mar): {leaky_score:.3f}")
print("Leakage check: the leaky score should be suspiciously high.")


honest accuracy (5 features): 0.956
leaky accuracy (+ March clk_mar): 1.000
Leakage check: the leaky score should be suspiciously high.


In [7]:

# Remove the leaked column. The final feature set contains exactly five features.
honest_feature_frame = model_df[
    ["client_hash_id","content_hash_id"] + feature_cols + ["went_dark"]
].copy()

assert "clk_mar" not in honest_feature_frame.columns
assert len(feature_cols) == 5

print("Removed leaky column: clk_mar")
print("Final model features:", feature_cols)
print(f"honest score kept: {honest_score:.3f}")


Removed leaky column: clk_mar
Final model features: ['log_imp_feb', 'log_clk_feb', 'ctr_feb', 'pos_feb', 'days_with_imps_feb']
honest score kept: 0.956


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** the warehouse is an unbalanced panel: some clients do not have a full February history. That means February-derived features can have different amounts of observed history across clients, so comparisons across pages are not perfectly uniform.

A second limitation is that `dim_content` is a snapshot. Fields that may have changed after 2026-02-28 must not be treated as historical February state unless their date proves they were already true by the cutoff.

The `gsc_data_available` flag also matters: a row with unavailable GSC data is **not** evidence of zero traffic, so unavailable rows are excluded from the GSC aggregates rather than converted into zeros.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.